# スポットレート予測 GBDT モデル - 探索ノートブック v1

## 目的
- OIS スポットレートの 3日後・5日後の変化量（bp）を LightGBM で予測
- 全テナーをプーリングして 1モデルで学習
- ウォークフォワード検証で OOS IC を計算

## 予測対象テナー
- 短期: 3M, 6M, 9M, 1Y
- 中長期: 2Y〜40Y (全テナー)

## データソース (全て DB)
- `irs_data`: OIS レート (2014-06-23〜)
- `boj_meetings`: BOJ 会合日・政策金利
- `foreign_yields`: US 国債 2Y/5Y/10Y/30Y + Fed Funds Rate
- `exchange_rates`: USDJPY, DXY
- `stock_prices`: 日経 225

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import bisect
import pickle
from scipy.stats import spearmanr

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

import lightgbm as lgb

from data.utils.database_manager import DatabaseManager

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup complete.')

## 1. 設定定数

In [ ]:
# --- 予測対象テナー ---
TARGET_TENORS_SHORT = ['3M', '6M', '9M', '1Y']
TARGET_TENORS_LONG  = [
    '2Y', '3Y', '4Y', '5Y', '6Y', '7Y', '8Y', '9Y',
    '10Y', '11Y', '12Y', '15Y', '20Y', '25Y', '30Y', '35Y', '40Y'
]
TARGET_TENORS = TARGET_TENORS_SHORT + TARGET_TENORS_LONG
print(f'Target tenors: {len(TARGET_TENORS)} series')
print(TARGET_TENORS)

# --- US 外債キーテナー ---
US_YIELD_TENORS = ['2Y', '5Y', '10Y', '30Y']

# --- データ期間 ---
DATA_START = '2014-06-23'   # OIS データ開始日
OOS_START  = '2020-01-01'   # OOS（アウトオブサンプル）開始日

# --- ウォークフォワード設定 ---
TEST_WINDOW_DAYS = 90   # テストウィンドウ（日数）
PURGE_DAYS       = 5    # パージ（ルックアヘッドリーク防止）
N_BOOST_ROUNDS   = 100  # 固定ラウンド数 (early stopping なし)

# --- 分数階差パラメータ ---
FRAC_D      = 0.4
FRAC_WINDOW = 50

# --- LightGBM パラメータ ---
LGBM_PARAMS = {
    'objective'       : 'regression',
    'metric'          : 'rmse',
    'verbosity'       : -1,
    'boosting_type'   : 'gbdt',
    'random_state'    : 42,
    'learning_rate'   : 0.05,
    'num_leaves'      : 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq'    : 5,
    'min_data_in_leaf': 30,
    'lambda_l2'       : 1.0,
}

## 2. データ読み込み

In [ ]:
db = DatabaseManager()

# ── 2-1. OIS レート ────────────────────────────────────
tenor_list = "', '".join(TARGET_TENORS)
ois_rows = db.select_as_dict(f"""
    SELECT trade_date, tenor, rate
    FROM irs_data
    WHERE product_type = 'OIS'
      AND tenor IN ('{tenor_list}')
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date, tenor
""")
df_ois_long = pd.DataFrame(ois_rows)
df_ois_long['trade_date'] = pd.to_datetime(df_ois_long['trade_date'])
df_ois_long['rate'] = pd.to_numeric(df_ois_long['rate'], errors='coerce').astype(float)

# ワイド形式に変換 (date × tenor)
df_ois = df_ois_long.pivot(index='trade_date', columns='tenor', values='rate')
df_ois = df_ois.reindex(columns=TARGET_TENORS)  # テナー順を固定
df_ois.columns = [f'OIS_{c}' for c in df_ois.columns]
print(f'OIS wide shape: {df_ois.shape}')
print(f'  期間: {df_ois.index.min().date()} ～ {df_ois.index.max().date()}')
print(f'  欠損率(列): {df_ois.isna().mean().round(3).to_dict()}')

In [ ]:
# ── 2-2. BOJ 会合・政策金利 ──────────────────────────────
mpm_rows = db.select_as_dict("""
    SELECT meeting_date, policy_rate_before, policy_rate_after, decision
    FROM boj_meetings
    ORDER BY meeting_date
""")
df_mpm = pd.DataFrame(mpm_rows)
df_mpm['meeting_date'] = pd.to_datetime(df_mpm['meeting_date'])
df_mpm['policy_rate_after'] = pd.to_numeric(df_mpm['policy_rate_after'], errors='coerce').astype(float)
print(f'BOJ meetings: {len(df_mpm)} rows')
print(df_mpm[df_mpm['decision'] != 'hold'][['meeting_date', 'policy_rate_before', 'policy_rate_after', 'decision']])

In [ ]:
# ── 2-3. US 国債 + Fed Funds Rate ──────────────────────────
us_tenor_list = "', '".join(US_YIELD_TENORS)
ust_rows = db.select_as_dict(f"""
    SELECT trade_date, tenor, yield_value
    FROM foreign_yields
    WHERE region = 'US'
      AND instrument_type = 'sovereign'
      AND tenor IN ('{us_tenor_list}')
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date, tenor
""")
df_ust_long = pd.DataFrame(ust_rows)
df_ust_long['trade_date'] = pd.to_datetime(df_ust_long['trade_date'])
df_ust_long['yield_value'] = pd.to_numeric(df_ust_long['yield_value'], errors='coerce').astype(float)

df_ust = df_ust_long.pivot(index='trade_date', columns='tenor', values='yield_value')
df_ust = df_ust.reindex(columns=US_YIELD_TENORS)
df_ust.columns = [f'UST_{c}' for c in df_ust.columns]

# Fed Funds Rate
ffr_rows = db.select_as_dict(f"""
    SELECT trade_date, yield_value as ffr
    FROM foreign_yields
    WHERE region = 'US' AND instrument_type = 'policy'
      AND tenor = 'ON'
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date
""")
df_ffr = pd.DataFrame(ffr_rows)
df_ffr['trade_date'] = pd.to_datetime(df_ffr['trade_date'])
df_ffr['ffr'] = pd.to_numeric(df_ffr['ffr'], errors='coerce').astype(float)
df_ffr = df_ffr.set_index('trade_date')

print(f'UST wide shape: {df_ust.shape}')
print(f'Fed Funds Rate rows: {len(df_ffr)}')

In [ ]:
# ── 2-4. FX (USDJPY, DXY) ────────────────────────────────
fx_rows = db.select_as_dict(f"""
    SELECT trade_date, currency_pair, close_price
    FROM exchange_rates
    WHERE currency_pair IN ('USDJPY', 'DXY')
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date
""")
df_fx_long = pd.DataFrame(fx_rows)
df_fx_long['trade_date'] = pd.to_datetime(df_fx_long['trade_date'])
df_fx_long['close_price'] = pd.to_numeric(df_fx_long['close_price'], errors='coerce').astype(float)

df_fx = df_fx_long.pivot(index='trade_date', columns='currency_pair', values='close_price')
print(f'FX shape: {df_fx.shape}')
print(df_fx.isna().sum())

# ── 2-5. 日経 225 ─────────────────────────────────────────
nk_rows = db.select_as_dict(f"""
    SELECT trade_date, close_price as nikkei
    FROM stock_prices
    WHERE ticker = '^N225'
      AND trade_date >= '{DATA_START}'
    ORDER BY trade_date
""")
df_nk = pd.DataFrame(nk_rows)
df_nk['trade_date'] = pd.to_datetime(df_nk['trade_date'])
df_nk['nikkei'] = pd.to_numeric(df_nk['nikkei'], errors='coerce').astype(float)
df_nk = df_nk.set_index('trade_date')

print(f'Nikkei rows: {len(df_nk)}')

## 3. ワイドデータ結合

In [ ]:
# OIS を基準に left join
df = df_ois.copy()
df = df.join(df_ust,  how='left')
df = df.join(df_ffr,  how='left')
df = df.join(df_fx,   how='left')
df = df.join(df_nk,   how='left')

print(f'Merged wide shape: {df.shape}')
print(f'期間: {df.index.min().date()} ～ {df.index.max().date()}')
print()

# 欠損状況の確認
missing = df.isna().mean().round(3)
print('=== 欠損率 ===')
print(missing[missing > 0])
# US・DXY の休場日（日本は営業日だが US が休み等）を前日値で補完
# → frac_diff window=50 が NaN の連鎖汚染を起こすのを防ぐ
# limit=5: 5日以内の連続欠損のみ補完（長期欠損は本物の欠損として残す）
ffill_cols = (
    [c for c in df.columns if c.startswith('UST_')] +
    ['ffr', 'USDJPY', 'DXY', 'nikkei']
)
ffill_cols = [c for c in ffill_cols if c in df.columns]
df[ffill_cols] = df[ffill_cols].ffill(limit=5)

print('=== ffill後の欠損率 ===')
missing_after = df.isna().mean().round(4)
print(missing_after[missing_after > 0])


## 4. BOJ カレンダー特徴量

- `policy_rate`: 各営業日時点での有効政策金利 (直近会合後のレート)
- `days_to_next_mpm`: 次回 BOJ 会合までの日数

In [ ]:
def build_boj_calendar_features(date_index: pd.DatetimeIndex, df_mpm: pd.DataFrame) -> pd.DataFrame:
    """
    各営業日について BOJ カレンダー関連の特徴量を生成する。

    Parameters
    ----------
    date_index : DatetimeIndex  全営業日
    df_mpm     : DataFrame      boj_meetings テーブルのデータ

    Returns
    -------
    DataFrame with columns:
        policy_rate       : 有効政策金利
        days_to_next_mpm  : 次回会合まで日数（カレンダー日）
    """
    meeting_dates = sorted(df_mpm['meeting_date'].tolist())
    rate_map = dict(zip(df_mpm['meeting_date'], df_mpm['policy_rate_after']))

    results = []
    for d in date_index:
        # 有効政策金利: d 以前の直近会合後レート
        past = [m for m in meeting_dates if m <= d]
        if past:
            eff_rate = rate_map[past[-1]]
        else:
            eff_rate = np.nan

        # 次回会合まで日数
        future = [m for m in meeting_dates if m > d]
        if future:
            days_to_next = (future[0] - d).days
        else:
            days_to_next = np.nan

        results.append({'policy_rate': eff_rate, 'days_to_next_mpm': days_to_next})

    return pd.DataFrame(results, index=date_index)


df_cal = build_boj_calendar_features(df.index, df_mpm)
df = df.join(df_cal)

print('Policy rate サンプル:')
print(df[['policy_rate', 'days_to_next_mpm']].describe())
print()
# 政策金利変化点の確認
policy_changes = df['policy_rate'].diff().fillna(0)
print('政策金利変化日:')
print(df[policy_changes != 0][['policy_rate']].head(10))

## 5. 特徴量エンジニアリング

### 5-1. 分数階差 (Fractional Differencing)
定常化しつつ過去情報を保持。d=0.4, window=50

In [ ]:
def frac_diff(series: pd.Series, d: float = 0.4, window: int = 50) -> pd.Series:
    """
    分数階差: w_k = (-1)^k * C(d, k)
    先頭 window-1 行は NaN（ウォームアップ期間）
    """
    weights = [1.0]
    for k in range(1, window):
        weights.append(-weights[-1] * (d - k + 1) / k)
    weights = np.array(weights[::-1])

    arr = series.to_numpy(dtype=float)
    n = len(arr)
    result = np.full(n, np.nan)
    if n < window:
        return pd.Series(result, index=series.index)

    from numpy.lib.stride_tricks import sliding_window_view
    windows_view = sliding_window_view(arr, window_shape=window)
    dots = windows_view @ weights
    dots[np.isnan(windows_view).any(axis=1)] = np.nan
    result[window - 1:] = dots
    return pd.Series(result, index=series.index)


# OIS スプレッド (OIS_tenor - policy_rate)
ois_cols = [c for c in df.columns if c.startswith('OIS_')]
for col in ois_cols:
    df[f'SPD_{col}'] = df[col] - df['policy_rate']

# 分数階差を適用
fd_source_cols = (
    [f'SPD_{c}' for c in ois_cols] +     # OIS スプレッド
    [c for c in df.columns if c.startswith('UST_')] +  # US 国債
    ['ffr', 'USDJPY', 'DXY']              # Fed Funds, FX
)

fd_cols_created = []
for col in fd_source_cols:
    if col in df.columns:
        fd_col = f'FD_{col}'
        df[fd_col] = frac_diff(df[col], d=FRAC_D, window=FRAC_WINDOW)
        fd_cols_created.append(fd_col)

# 日経: log return
df['nikkei_logret'] = np.log(df['nikkei'] / df['nikkei'].shift(1))

print(f'分数階差列 {len(fd_cols_created)} 個作成')
print('サンプル (最初の60行はNaN):')
print(df[fd_cols_created[:3]].tail(5))

In [ ]:
# カーブ形状スプレッド（OIS内・US内）
# OIS: キーポイント間スプレッド
ois_key = ['OIS_2Y', 'OIS_5Y', 'OIS_10Y', 'OIS_20Y', 'OIS_30Y']
for col in ois_key:
    if col in df.columns:
        pass  # 実水準は既にある

df['OIS_10Y2Y_spd'] = df['OIS_10Y'] - df['OIS_2Y']
df['OIS_30Y10Y_spd'] = df['OIS_30Y'] - df['OIS_10Y']
df['OIS_5Y2Y_spd']  = df['OIS_5Y']  - df['OIS_2Y']

# US: キーポイント間スプレッド
df['UST_10Y2Y_spd']  = df['UST_10Y']  - df['UST_2Y']
df['UST_30Y10Y_spd'] = df['UST_30Y']  - df['UST_10Y']

# JPN - US スプレッド (10Y同士)
df['JPUS_10Y_spd'] = df['OIS_10Y'] - df['UST_10Y']
df['JPUS_2Y_spd']  = df['OIS_2Y']  - df['UST_2Y']

# これらにも分数階差を適用
curve_spd_cols = ['OIS_10Y2Y_spd', 'OIS_30Y10Y_spd', 'OIS_5Y2Y_spd',
                  'UST_10Y2Y_spd', 'UST_30Y10Y_spd',
                  'JPUS_10Y_spd', 'JPUS_2Y_spd']
for col in curve_spd_cols:
    df[f'FD_{col}'] = frac_diff(df[col], d=FRAC_D, window=FRAC_WINDOW)

print('スプレッド列 作成完了')
print(df[curve_spd_cols].tail(3))

### 5-2. 欠損補完 (MICE)

OIS 各テナーの欠損を IterativeImputer + BayesianRidge で補完。  
補完前に `_is_imputed` フラグを付け、モデルへ渡す。

In [ ]:
# ── MICE 欠損補完 ────────────────────────────────────────────────────────────
#
# 【設計原則】
# 1. 「多変量」で一気に解く:
#    OIS 全テナー + 外部指標（UST/FFR/FX/Nikkei）を一つの行列として投入。
#    「ドル円が動いたから欠損している金利もこれくらい動いているはず」という
#    多変量間の相関を MICE が学習できる。OIS 単独では上記が不可能。
#
# 2. BayesianRidge を使う:
#    ノイズの多い金融データでも過学習を抑え、自然な値を推定できる。
#
# 3. 補完前に is_imputed フラグを作る:
#    後続の LightGBM に「ここは補完されたデータ」と伝えることで予測精度を維持。

# 補完対象: OIS 全テナー（欠損あり）
ois_raw_cols = [c for c in ois_cols]   # ['OIS_3M', 'OIS_6M', ..., 'OIS_40Y']
print(f'補完対象 OIS 列: {len(ois_raw_cols)} 系列')
print('補完前の欠損数:')
missing_before = df[ois_raw_cols].isna().sum()
print(missing_before[missing_before > 0])

# ── Step 1: 補完前フラグ（OIS 列のみ）─────────────────────────────────────
for col in ois_raw_cols:
    df[f'{col}_is_imputed'] = df[col].isna().astype(int)

total_imputed = sum(df[f'{col}_is_imputed'].sum() for col in ois_raw_cols)
print(f'\n補完が必要なセル数: {total_imputed}')

# ── Step 2: アンカー列を定義（欠損がほぼなく OIS と相関が高い外部指標）──────
# これを一緒に投入することで MICE が多変量相関を利用した自然な補完が可能になる
anchor_cols = [
    'UST_2Y', 'UST_5Y', 'UST_10Y', 'UST_30Y',   # US 国債（欠損ほぼなし）
    'ffr',                                         # Fed Funds Rate
    'USDJPY', 'DXY',                               # FX
    'nikkei',                                      # 日経 225
]
anchor_cols = [c for c in anchor_cols if c in df.columns]
print(f'\nアンカー列 ({len(anchor_cols)}): {anchor_cols}')
print('アンカー列の欠損率:')
print(df[anchor_cols].isna().mean().round(4))

# ── Step 3: MICE 実行 ─────────────────────────────────────────────────────
# 補完行列 = OIS 全テナー + アンカー列（多変量一括）
impute_target_cols = ois_raw_cols + anchor_cols

imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    imputation_order='ascending',   # 欠損の少ない列（アンカー）から順に補完
    verbose=0,
)

# 全期間で fit_transform（金利間の相関構造は長期安定のため許容）
print('\nMICE 実行中... (数秒かかります)')
imputed_matrix = imputer.fit_transform(df[impute_target_cols])
imputed_df = pd.DataFrame(imputed_matrix, index=df.index, columns=impute_target_cols)

# OIS 列のみ df に書き戻す（アンカー列は元の値のまま維持）
df[ois_raw_cols] = imputed_df[ois_raw_cols]

print('補完後の欠損数:')
remaining = df[ois_raw_cols].isna().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '欠損なし（全補完完了）')
print('\nMICE 完了')

# ── Step 4: 補完結果の可視化（3M テナーの例）──────────────────────────────
if 'OIS_3M' in df.columns and df['OIS_3M_is_imputed'].sum() > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    imputed_mask = df['OIS_3M_is_imputed'] == 1
    ax.plot(df.index, df['OIS_3M'], color='steelblue', label='OIS 3M（補完済み）', alpha=0.8)
    ax.scatter(df.index[imputed_mask], df.loc[imputed_mask, 'OIS_3M'],
               color='orange', s=12, zorder=5, label='MICE 補完点', alpha=0.9)
    ax.set_title('OIS 3M: MICE 補完結果確認（橙色 = 補完箇所）')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'OIS_3M の補完点数: {imputed_mask.sum()}')
else:
    print('OIS_3M に欠損なし (補完不要)')


## 6. プーリング: ワイド → ロング形式

複数テナーを縦積みして 1モデルで学習。

In [ ]:
# OIS 実水準列（= プーリングの rate_value）
# テナー名 -> インデックスのマッピング
label_to_index = {tenor: i for i, tenor in enumerate(TARGET_TENORS)}
print('Tenor index mapping:')
print(label_to_index)

# ワイドの「OIS_xxx」以外の列（共通特徴量）
# プーリング対象外の共通特徴量列
id_cols = [
    'policy_rate', 'days_to_next_mpm',
    # UST
    'UST_2Y', 'UST_5Y', 'UST_10Y', 'UST_30Y',
    'FD_SPD_OIS_2Y', 'FD_SPD_OIS_5Y', 'FD_SPD_OIS_10Y', 'FD_SPD_OIS_30Y',  # 代表テナーのFD
    'ffr',
    'USDJPY', 'DXY', 'nikkei', 'nikkei_logret',
    'FD_UST_2Y', 'FD_UST_5Y', 'FD_UST_10Y', 'FD_UST_30Y',
    'FD_ffr', 'FD_USDJPY', 'FD_DXY',
    'OIS_10Y2Y_spd', 'OIS_30Y10Y_spd', 'OIS_5Y2Y_spd',
    'UST_10Y2Y_spd', 'UST_30Y10Y_spd',
    'JPUS_10Y_spd', 'JPUS_2Y_spd',
    'FD_OIS_10Y2Y_spd', 'FD_OIS_30Y10Y_spd', 'FD_OIS_5Y2Y_spd',
    'FD_UST_10Y2Y_spd', 'FD_UST_30Y10Y_spd',
    'FD_JPUS_10Y_spd', 'FD_JPUS_2Y_spd',
]
# 存在する列のみフィルタ
id_cols = [c for c in id_cols if c in df.columns]
print(f'\n共通特徴量 (id_cols): {len(id_cols)} 列')

# プーリング対象の rate 列（OIS 実水準）+ is_imputed フラグ
rate_value_cols = [f'OIS_{t}' for t in TARGET_TENORS]
imputed_flag_cols = [f'OIS_{t}_is_imputed' for t in TARGET_TENORS if f'OIS_{t}_is_imputed' in df.columns]

# DF にインデックスを列化してリセット
df_reset = df.reset_index().rename(columns={'trade_date': 'Date'})

# melt でロング変換
pooled = df_reset.melt(
    id_vars=['Date'] + id_cols,
    value_vars=rate_value_cols,
    var_name='Rate_Label',
    value_name='Rate_Value',
)
pooled['Rate_Label'] = pooled['Rate_Label'].str.replace('OIS_', '', regex=False)
pooled['Series_Index'] = pooled['Rate_Label'].map(label_to_index).astype('category')

# FD_SPD (テナー別) をロング形式でマージ
# 各テナーの SPD と FD_SPD を個別に付与
spd_cols_wide = {t: f'SPD_OIS_{t}' for t in TARGET_TENORS if f'SPD_OIS_{t}' in df.columns}
fd_spd_cols_wide = {t: f'FD_SPD_OIS_{t}' for t in TARGET_TENORS if f'FD_SPD_OIS_{t}' in df.columns}

df_spd = df_reset[['Date'] + list(spd_cols_wide.values())].melt(
    id_vars=['Date'], value_vars=list(spd_cols_wide.values()),
    var_name='_col', value_name='SPD_to_policy'
)
df_spd['Rate_Label'] = df_spd['_col'].str.replace('SPD_OIS_', '', regex=False)
df_spd = df_spd.drop(columns='_col')

df_fd_spd = df_reset[['Date'] + list(fd_spd_cols_wide.values())].melt(
    id_vars=['Date'], value_vars=list(fd_spd_cols_wide.values()),
    var_name='_col', value_name='FD_SPD_to_policy'
)
df_fd_spd['Rate_Label'] = df_fd_spd['_col'].str.replace('FD_SPD_OIS_', '', regex=False)
df_fd_spd = df_fd_spd.drop(columns='_col')

# is_imputed フラグ
df_imp = df_reset[['Date'] + imputed_flag_cols].melt(
    id_vars=['Date'], value_vars=imputed_flag_cols,
    var_name='_col', value_name='is_imputed'
)
df_imp['Rate_Label'] = df_imp['_col'].str.replace('OIS_', '').str.replace('_is_imputed', '')
df_imp = df_imp.drop(columns='_col')

# 結合
pooled = pooled.merge(df_spd,    on=['Date', 'Rate_Label'], how='left')
pooled = pooled.merge(df_fd_spd, on=['Date', 'Rate_Label'], how='left')
pooled = pooled.merge(df_imp,    on=['Date', 'Rate_Label'], how='left')

pooled = pooled.sort_values(['Date', 'Series_Index']).reset_index(drop=True)

print(f'\nPooled DataFrame shape: {pooled.shape}')
print(f'Columns: {pooled.columns.tolist()}')
print(pooled.head(5))

## 7. 目的変数の生成

3日後・5日後の変化量（%単位）→ bp換算 → 系列別std正規化

In [ ]:
# 系列ごとに shift(h) で h 日後の Rate_Value を計算
pooled = pooled.sort_values(['Rate_Label', 'Date']).reset_index(drop=True)

for h in [3, 5]:
    # h 営業日後のレート（Rate_Label ごとに shift）
    pooled[f'Rate_Value_t{h}'] = (
        pooled.groupby('Rate_Label')['Rate_Value']
              .shift(-h)
    )
    # 変化量 (% 単位)
    pooled[f'Target_{h}d'] = pooled[f'Rate_Value_t{h}'] - pooled['Rate_Value']
    # 系列別全期間 std で正規化 (異分散補正)
    instr_std = pooled.groupby('Rate_Label')[f'Target_{h}d'].transform('std')
    pooled[f'Target_{h}d_std']  = instr_std
    pooled[f'Target_{h}d_norm'] = pooled[f'Target_{h}d'] / instr_std

# 日付ソート順に戻す
pooled = pooled.sort_values(['Date', 'Series_Index']).reset_index(drop=True)

print('Target distribution (3d, 5d):')
print(pooled[['Target_3d', 'Target_5d']].describe())
print()
print('Target std by tenor (3d):')
print(pooled.groupby('Rate_Label')['Target_3d'].std().sort_index())

## 8. 特徴量カラム定義

In [ ]:
# 学習に使わない列（除外リスト: ブラックリスト方式）
NON_FEATURE_COLS = {
    'Date', 'Rate_Label', 'Rate_Value',
    'Rate_Value_t3', 'Rate_Value_t5',
    'Target_3d', 'Target_3d_norm', 'Target_3d_std',
    'Target_5d', 'Target_5d_norm', 'Target_5d_std',
}

feature_cols = [c for c in pooled.columns if c not in NON_FEATURE_COLS]
print(f'特徴量数: {len(feature_cols)}')
print(feature_cols)

# カテゴリ列の確認
cat_cols = [c for c in feature_cols if pooled[c].dtype.name == 'category']
print(f'\nカテゴリ列: {cat_cols}')

## 9. データ品質チェック

ウォークフォワード前の最終確認

In [ ]:
# 欠損確認（特徴量）
feat_missing = pooled[feature_cols].isna().mean().sort_values(ascending=False)
print('=== 特徴量欠損率 (上位20件) ===')
print(feat_missing[feat_missing > 0].head(20))

# OOS 開始以降のデータ確認
oos_data = pooled[pooled['Date'] >= OOS_START]
print(f'\nOOS データ: {len(oos_data)} rows')
print(f'  期間: {oos_data["Date"].min().date()} ～ {oos_data["Date"].max().date()}')
print(f'  日数: {oos_data["Date"].nunique()} 営業日')
print(f'  系列数: {oos_data["Rate_Label"].nunique()}')

In [ ]:
# OIS カーブの時系列プロット (代表テナー)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

key_tenors = ['3M', '1Y', '2Y', '5Y', '10Y', '30Y']
df_plot = pooled[pooled['Rate_Label'].isin(key_tenors)].copy()
df_plot_pivot = df_plot.pivot(index='Date', columns='Rate_Label', values='Rate_Value')
df_plot_pivot = df_plot_pivot.reindex(columns=key_tenors)

for col in key_tenors:
    if col in df_plot_pivot.columns:
        axes[0].plot(df_plot_pivot.index, df_plot_pivot[col], label=col, alpha=0.8)
axes[0].set_title('OIS スポットレート (代表テナー)')
axes[0].set_ylabel('Rate (%)')
axes[0].legend(ncol=3)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

# 政策金利
axes[1].plot(df['policy_rate'].dropna(), label='BOJ Policy Rate', color='red', linewidth=2)
axes[1].plot(df['UST_10Y'].dropna(), label='UST 10Y', color='blue', alpha=0.7)
axes[1].set_title('BOJ 政策金利 / US10Y')
axes[1].set_ylabel('Rate (%)')
axes[1].legend()
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('/Users/nishiharahiroto/Documents/programs/market-analytics-ver1/analysis/ml/ois_rates_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('プロット保存完了')

## 10. ウォークフォワード検証

拡張窓 (expanding window) + パージ付きウォークフォワード

In [ ]:
def train_lgbm(X_train, y_train, cat_cols_idx, params, n_rounds):
    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols_idx, free_raw_data=False)
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=n_rounds,
        valid_sets=None,
    )
    return model


def walk_forward_validation(pooled_df, feature_cols, target_col, horizon_label,
                            oos_start, test_window_days, purge_days,
                            params, n_rounds, cat_col_names):
    """
    拡張窓ウォークフォワード検証。

    Parameters
    ----------
    target_col    : 正規化済み目的変数列名 (e.g., 'Target_3d_norm')
    horizon_label : ラベル (e.g., '3d')
    cat_col_names : カテゴリ変数列名のリスト
    """
    df_clean = pooled_df.dropna(subset=feature_cols + [target_col]).copy()
    dates = df_clean['Date'].values
    unique_dates = sorted(df_clean['Date'].unique())

    # OOS 開始インデックス
    oos_start_dt = pd.Timestamp(oos_start)
    test_start_idx = bisect.bisect_left(unique_dates, oos_start_dt)
    if test_start_idx >= len(unique_dates):
        raise ValueError(f'OOS start {oos_start} is beyond data range')

    X_all = df_clean[feature_cols].values
    y_all = df_clean[target_col].values
    labels_all = df_clean['Rate_Label'].values

    # カテゴリ列のインデックス
    cat_cols_idx = [feature_cols.index(c) for c in cat_col_names if c in feature_cols]

    results = []
    fold = 0
    current_idx = test_start_idx

    while current_idx < len(unique_dates):
        train_end  = unique_dates[current_idx - 1]
        test_start = unique_dates[current_idx]
        purge_end  = train_end - pd.Timedelta(days=purge_days)
        test_end   = test_start + pd.Timedelta(days=test_window_days)

        train_mask = df_clean['Date'] <= purge_end
        test_mask  = (df_clean['Date'] >= test_start) & (df_clean['Date'] < test_end)

        if train_mask.sum() < 500 or test_mask.sum() < 10:
            break

        X_train, y_train = X_all[train_mask], y_all[train_mask]
        X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

        model = train_lgbm(X_train, y_train, cat_cols_idx, params, n_rounds)
        preds = model.predict(X_test)
        train_preds = model.predict(X_train)
        train_ic, _ = spearmanr(y_train, train_preds)

        fold_result = pd.DataFrame({
            'Fold'      : fold,
            'Date'      : df_clean.loc[test_mask, 'Date'].values,
            'Rate_Label': labels_all[test_mask],
            'Actual'    : y_test,
            'Pred'      : preds,
            'Train_IC'  : train_ic,
        })
        results.append(fold_result)

        fold += 1
        next_date = test_start + pd.Timedelta(days=test_window_days)
        current_idx = bisect.bisect_left(unique_dates, next_date)
        if current_idx >= len(unique_dates):
            break

    results_df = pd.concat(results).reset_index(drop=True)
    print(f'  [{horizon_label}] Folds: {fold}, OOS rows: {len(results_df)}')
    return results_df, model  # 最終モデルを返す（特徴量重要度確認用）


print('Walk-forward 関数定義完了')

In [ ]:
def summarize_ic(results_df, horizon_label, recent_n_folds=3):
    """IC サマリーを計算して表示"""
    # Global IC
    ic_all, _ = spearmanr(results_df['Actual'], results_df['Pred'])

    # CS IC (Cross-Sectional)
    cs_ics = []
    for date, grp in results_df.groupby('Date'):
        if len(grp) >= 3:
            ic, _ = spearmanr(grp['Actual'], grp['Pred'])
            if not np.isnan(ic):
                cs_ics.append(ic)
    cs_ic = np.mean(cs_ics)

    # TS IC (Time-Series per tenor)
    ts_ics = []
    for label, grp in results_df.groupby('Rate_Label'):
        if len(grp) >= 5:
            ic, _ = spearmanr(grp['Actual'], grp['Pred'])
            if not np.isnan(ic):
                ts_ics.append(ic)
    ts_ic = np.mean(ts_ics)

    # Train IC
    train_ic = results_df.groupby('Fold')['Train_IC'].first().mean()
    gap = train_ic - ic_all

    # 直近 N フォールド IC
    recent_folds = results_df['Fold'].max() - recent_n_folds + 1
    recent_data = results_df[results_df['Fold'] >= recent_folds]
    ic_recent, _ = spearmanr(recent_data['Actual'], recent_data['Pred'])

    # フォールド別 IC
    ic_by_fold = {}
    for fold, grp in results_df.groupby('Fold'):
        ic, _ = spearmanr(grp['Actual'], grp['Pred'])
        ic_by_fold[fold] = round(ic, 4)

    print(f'\n=== {horizon_label} OOS IC Summary ===')
    print(f'  Global IC  : {ic_all:.4f}')
    print(f'  CS IC      : {cs_ic:.4f}')
    print(f'  TS IC      : {ts_ic:.4f}')
    print(f'  Train IC   : {train_ic:.4f}')
    print(f'  Gap        : {gap:.4f}  (Train - Global; < 0.20 が目安)')
    print(f'  IC Recent  : {ic_recent:.4f}  (直近 {recent_n_folds} フォールド)')
    print(f'  IC by Fold : {ic_by_fold}')

    return {
        'ic_all': ic_all, 'cs_ic': cs_ic, 'ts_ic': ts_ic,
        'train_ic': train_ic, 'gap': gap,
        'ic_recent': ic_recent, 'ic_by_fold': ic_by_fold
    }

print('IC 計算関数定義完了')

In [ ]:
# ウォークフォワード実行 (3d)
print('=== Walk-Forward: 3日後予測 ===')
results_3d, last_model_3d = walk_forward_validation(
    pooled_df       = pooled,
    feature_cols    = feature_cols,
    target_col      = 'Target_3d_norm',
    horizon_label   = '3d',
    oos_start       = OOS_START,
    test_window_days= TEST_WINDOW_DAYS,
    purge_days      = PURGE_DAYS,
    params          = LGBM_PARAMS,
    n_rounds        = N_BOOST_ROUNDS,
    cat_col_names   = cat_cols,
)
summary_3d = summarize_ic(results_3d, '3d')

In [ ]:
# ウォークフォワード実行 (5d)
print('=== Walk-Forward: 5日後予測 ===')
results_5d, last_model_5d = walk_forward_validation(
    pooled_df       = pooled,
    feature_cols    = feature_cols,
    target_col      = 'Target_5d_norm',
    horizon_label   = '5d',
    oos_start       = OOS_START,
    test_window_days= TEST_WINDOW_DAYS,
    purge_days      = PURGE_DAYS,
    params          = LGBM_PARAMS,
    n_rounds        = N_BOOST_ROUNDS,
    cat_col_names   = cat_cols,
)
summary_5d = summarize_ic(results_5d, '5d')

## 11. 結果分析

In [ ]:
# テナー別 TS IC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, res, label in [(axes[0], results_3d, '3d'), (axes[1], results_5d, '5d')]:
    ts_by_tenor = {}
    for tenor, grp in res.groupby('Rate_Label'):
        if len(grp) >= 5:
            ic, _ = spearmanr(grp['Actual'], grp['Pred'])
            ts_by_tenor[tenor] = ic

    tenor_order = [t for t in TARGET_TENORS if t in ts_by_tenor]
    ic_vals = [ts_by_tenor[t] for t in tenor_order]
    colors = ['red' if v < 0 else 'steelblue' for v in ic_vals]

    ax.bar(range(len(tenor_order)), ic_vals, color=colors)
    ax.set_xticks(range(len(tenor_order)))
    ax.set_xticklabels(tenor_order, rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_title(f'TS IC by Tenor ({label})')
    ax.set_ylabel('Spearman IC')

plt.tight_layout()
plt.savefig('/Users/nishiharahiroto/Documents/programs/market-analytics-ver1/analysis/ml/ic_by_tenor.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# フォールド別 IC 推移
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, summ, label in [(axes[0], summary_3d, '3d'), (axes[1], summary_5d, '5d')]:
    folds = list(summ['ic_by_fold'].keys())
    ics   = list(summ['ic_by_fold'].values())
    ax.bar(folds, ics, color=['steelblue' if v >= 0 else 'red' for v in ics])
    ax.axhline(summ['ic_all'], color='orange', linestyle='--', label=f'Global IC={summ["ic_all"]:.3f}')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'IC by Fold ({label})')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Spearman IC')
    ax.legend()

plt.tight_layout()
plt.savefig('/Users/nishiharahiroto/Documents/programs/market-analytics-ver1/analysis/ml/ic_by_fold.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 特徴量重要度 (最終フォールドのモデル)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, model, label in [(axes[0], last_model_3d, '3d'), (axes[1], last_model_5d, '5d')]:
    imp = pd.Series(model.feature_importance(importance_type='gain'),
                    index=feature_cols).sort_values(ascending=False).head(30)
    imp.plot(kind='barh', ax=ax)
    ax.invert_yaxis()
    ax.set_title(f'Feature Importance - Gain ({label}, top 30)')
    ax.set_xlabel('Gain')

plt.tight_layout()
plt.savefig('/Users/nishiharahiroto/Documents/programs/market-analytics-ver1/analysis/ml/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ベースライン記録
print('=' * 60)
print('ベースライン結果 (v1: 全テナープーリング, MPM除外なし)')
print(f'start_date: {OOS_START}')
print(f'rounds: {N_BOOST_ROUNDS}')
print()

for label, summ in [('3d', summary_3d), ('5d', summary_5d)]:
    print(f'{label}: Global IC={summ["ic_all"]:.4f}, '
          f'CS IC={summ["cs_ic"]:.4f}, '
          f'TS IC={summ["ts_ic"]:.4f}, '
          f'Gap={summ["gap"]:.4f}, '
          f'ic_recent={summ["ic_recent"]:.4f}')
print('=' * 60)

## 12. 最新日の予測値確認

最新データで全テナーの 3d/5d 予測変化量を確認

In [ ]:
# 最新日のデータで予測
latest_date = pooled['Date'].max()
df_latest = pooled[pooled['Date'] == latest_date].copy()

print(f'予測基準日: {latest_date.date()}')
print(f'系列数: {len(df_latest)}')

# 欠損確認
missing_latest = df_latest[feature_cols].isna().sum()
if missing_latest.sum() > 0:
    print('\n欠損特徴量:')
    print(missing_latest[missing_latest > 0])

X_latest = df_latest[feature_cols].values

# 予測 (逆正規化)
pred_3d_norm = last_model_3d.predict(X_latest)
pred_5d_norm = last_model_5d.predict(X_latest)

# std は最新日の値
std_3d = df_latest['Target_3d_std'].values
std_5d = df_latest['Target_5d_std'].values

pred_3d_pct = pred_3d_norm * std_3d        # % 単位
pred_5d_pct = pred_5d_norm * std_5d        # % 単位
pred_3d_bp  = pred_3d_pct * 100            # bp 単位
pred_5d_bp  = pred_5d_pct * 100            # bp 単位

df_pred = pd.DataFrame({
    'Tenor'         : df_latest['Rate_Label'].values,
    'Current_Rate_%': df_latest['Rate_Value'].values,
    'Pred_3d_bp'    : pred_3d_bp.round(2),
    'Pred_5d_bp'    : pred_5d_bp.round(2),
    'Pred_3d_rate_%': (df_latest['Rate_Value'].values + pred_3d_pct).round(5),
    'Pred_5d_rate_%': (df_latest['Rate_Value'].values + pred_5d_pct).round(5),
})
df_pred = df_pred.set_index('Tenor').reindex(TARGET_TENORS)

print()
print(df_pred.to_string())